# Validação - Câmara/Brasil

Confere os 27 estados, o recorte de UF e os arquivos gerados.

Nesta versão a validação também diferencia registros e deputados únicos, verifica o fallback de proposições, resume a CEAP sem UF distribuível e marca o ano corrente como parcial.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re

import pandas as pd

ROOT = Path("/Volumes/workspace/pi_ii_bronze/camara")
ANOS = [2023, 2024, 2025, 2026]

UFS = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

CONTAGEM_COMPLETA = False

def pasta_uf(uf):
    return ROOT / uf.lower()

def normalizar(valor):
    return re.sub(r"[^a-z0-9]", "", str(valor).lower())

print("Raiz:", ROOT)


In [ ]:
# arquivos esperados por UF
nomes = ["deputados.csv"]

for ano in ANOS:
    nomes += [
        f"proposicoes_autores_{ano}.csv",
        f"proposicoes_{ano}.csv",
        f"proposicoes_fallback_{ano}.csv",
        f"despesas_{ano}.csv",
        f"votos_{ano}.csv",
        f"votacoes_{ano}.csv",
    ]

linhas = []

for uf in UFS:
    for nome in nomes:
        path = pasta_uf(uf) / nome

        linhas.append({
            "uf": uf,
            "arquivo": nome,
            "existe": path.exists(),
            "tamanho_mb": (
                round(path.stat().st_size / 1024**2, 2)
                if path.exists() else None
            ),
        })

arquivos = pd.DataFrame(linhas)
faltantes = arquivos.loc[~arquivos["existe"]].copy()

auditorias_despesas = []
for ano in ANOS:
    path = ROOT / f"_despesas_nao_distribuidas_{ano}.csv"
    auditorias_despesas.append({
        "ano": ano,
        "arquivo": path.name,
        "existe": path.exists(),
        "tamanho_mb": (
            round(path.stat().st_size / 1024**2, 2)
            if path.exists() else None
        ),
    })

auditorias_despesas_df = pd.DataFrame(auditorias_despesas)

print("Arquivos esperados por UF:", len(arquivos))
print("Faltantes:", len(faltantes))
print("Auditorias CEAP:", int(auditorias_despesas_df["existe"].sum()), "/", len(ANOS))

if len(faltantes):
    display(faltantes)

display(auditorias_despesas_df)


In [ ]:
# deputados: registros x IDs únicos
resumo_deputados = []

for uf in UFS:
    path = pasta_uf(uf) / "deputados.csv"
    if not path.exists():
        continue

    df = pd.read_csv(
        path,
        sep=";",
        dtype=str,
        keep_default_na=False,
    )

    id_col = None
    for c in df.columns:
        if normalizar(c) in {"id", "iddeputado", "deputadoid"}:
            id_col = c
            break

    if not id_col:
        raise RuntimeError(
            f"{uf}: não encontrei a coluna de ID em deputados.csv"
        )

    resumo_deputados.append({
        "uf": uf,
        "registros": len(df),
        "deputados_unicos": df[id_col].astype(str).nunique(),
        "registros_repetidos_por_id": (
            len(df) - df[id_col].astype(str).nunique()
        ),
    })

deputados_df = pd.DataFrame(resumo_deputados)

deputados_df.to_csv(
    ROOT / "_deputados_por_uf.csv",
    sep=";",
    index=False,
    encoding="utf-8",
)

display(deputados_df)


In [ ]:
# leitura rápida + checagem de UF
alertas = []

for uf in UFS:
    for ano in ANOS:
        for tipo in ("despesas", "votos"):
            path = pasta_uf(uf) / f"{tipo}_{ano}.csv"

            if not path.exists() or path.stat().st_size == 0:
                continue

            try:
                amostra = pd.read_csv(
                    path,
                    sep=";",
                    dtype=str,
                    nrows=3000,
                    keep_default_na=False,
                )
            except Exception as exc:
                alertas.append(
                    f"{uf} / {tipo}_{ano}: erro de leitura: {exc}"
                )
                continue

            if amostra.empty:
                continue

            candidatos = []
            for c in amostra.columns:
                nc = normalizar(c)

                if tipo == "despesas":
                    if nc in {"sguf", "siglauf", "uf"}:
                        candidatos.append(c)
                else:
                    if "deputado" in nc and "uf" in nc:
                        candidatos.append(c)

            if candidatos:
                col = candidatos[0]
                encontrados = set(
                    amostra[col]
                    .astype(str)
                    .str.strip()
                    .str.upper()
                )

                encontrados.discard("")

                if not encontrados.issubset({uf}):
                    alertas.append(
                        f"{uf} / {tipo}_{ano}: "
                        f"UF inesperada em {col}: {sorted(encontrados)}"
                    )

print("Alertas:", len(alertas))
for alerta in alertas[:100]:
    print(" -", alerta)


In [ ]:
# opcional: contagem completa dos CSVs
def contar(path):
    total = 0

    for chunk in pd.read_csv(
        path,
        sep=";",
        dtype=str,
        chunksize=200_000,
        keep_default_na=False,
    ):
        total += len(chunk)

    return total

contagens = []

if CONTAGEM_COMPLETA:
    for uf in UFS:
        for nome in nomes:
            path = pasta_uf(uf) / nome

            contagens.append({
                "uf": uf,
                "arquivo": nome,
                "registros": contar(path) if path.exists() else None,
            })

    contagens_df = pd.DataFrame(contagens)
    contagens_df.to_csv(
        ROOT / "_contagens_brasil.csv",
        sep=";",
        index=False,
        encoding="utf-8",
    )

    display(contagens_df)
else:
    print("Contagem completa desativada.")

# estes arquivos são pequenos; conta sempre
resumo_nao_distribuidas = []

for ano in ANOS:
    path = ROOT / f"_despesas_nao_distribuidas_{ano}.csv"

    resumo_nao_distribuidas.append({
        "ano": ano,
        "registros_nao_distribuidos": (
            contar(path) if path.exists() else None
        ),
    })

nao_distribuidas_df = pd.DataFrame(resumo_nao_distribuidas)
display(nao_distribuidas_df)


In [ ]:
# resumo por estado
resumo_estados = (
    arquivos
    .groupby("uf", as_index=False)
    .agg(
        arquivos_esperados=("arquivo", "count"),
        arquivos_presentes=("existe", "sum"),
        tamanho_total_mb=("tamanho_mb", "sum"),
    )
)

resumo_estados["completo"] = (
    resumo_estados["arquivos_esperados"]
    == resumo_estados["arquivos_presentes"]
)

resumo_estados = resumo_estados.merge(
    deputados_df[["uf", "registros", "deputados_unicos"]],
    on="uf",
    how="left",
)

resumo_estados.to_csv(
    ROOT / "resumo_brasil.csv",
    sep=";",
    index=False,
    encoding="utf-8",
)

ano_atual = datetime.now(timezone.utc).year
anos_parciais = [ano_atual] if ano_atual in ANOS else []

auditorias_ceap_faltantes = auditorias_despesas_df.loc[
    ~auditorias_despesas_df["existe"], "ano"
].tolist()

status = {
    "ufs": len(UFS),
    "anos": ANOS,
    "anos_parciais": anos_parciais,
    "arquivos_esperados": int(len(arquivos)),
    "arquivos_faltantes": int(len(faltantes)),
    "auditorias_ceap_faltantes": auditorias_ceap_faltantes,
    "alertas": alertas,
    "status": (
        "ok"
        if (
            len(faltantes) == 0
            and len(alertas) == 0
            and len(auditorias_ceap_faltantes) == 0
        )
        else "revisar"
    ),
    "validado_em_utc": datetime.now(timezone.utc).isoformat(),
}

with (ROOT / "_status_pipeline_brasil.json").open(
    "w", encoding="utf-8"
) as f:
    json.dump(status, f, ensure_ascii=False, indent=2)

display(resumo_estados)
print(json.dumps(status, ensure_ascii=False, indent=2))
